# 028 — Modelos ocultos de Markov

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Soluciones explicadas

**E1.** Predicción: `(0.7·0.5+0.3·0.5, 0.3·0.5+0.7·0.5) = (0.5, 0.5)`. Corrección: `(0.9·0.5, 0.2·0.5) = (0.45, 0.10)`; normalizando entre 0.55: **`P(lluvia₁|par₁) ≈ 0.818`**. El sensor manda porque la predicción era neutra.

**E2.** Predicción desde `(0.818, 0.182)`: `(0.7·0.818+0.3·0.182, …) = (0.627, 0.373)`. Corrección: `(0.9·0.627, 0.2·0.373) = (0.564, 0.0746)` → **`0.883`**. Sube pero cada vez menos: la transición (0.7/0.3) "mezcla" la creencia hacia el equilibrio y pone un techo — no se satura en 1.

**E3.** Corrección con `P(¬par|·) = (0.1, 0.8)`: `(0.1·0.627, 0.8·0.373) = (0.0627, 0.298)` → `P(lluvia₂) ≈ 0.174`. Una sola observación contraria revierte la creencia: con solo 2 estados y emisiones informativas, la evidencia reciente pesa mucho.

**E4.** La corrección `b(x) ∝ P(eₜ|x)·b'(x)` ya no factoriza: haría falta arrastrar `P(xₜ, xₜ₋₁ | e₁:ₜ)`, duplicando el estado (pares de estados) y elevando el costo a `O(|S|⁴)` por paso en el peor caso — o redefinir el estado del HMM para restaurar Markov.


In [ ]:
result = run_lab("probability", seed=28)
assert result["kind"] == "probability"
assert result["evidence"]
show(result)


In [ ]:
A = [[0.7, 0.3], [0.3, 0.7]]
B = {"par": [0.9, 0.2], "no": [0.1, 0.8]}

def forward_step(b, obs):
    pred = [sum(A[i][j]*b[i] for i in range(2)) for j in range(2)]
    corr = [B[obs][j]*pred[j] for j in range(2)]
    z = sum(corr)
    return [c/z for c in corr]

b1 = forward_step([0.5, 0.5], "par")
b2 = forward_step(b1, "par")
b2n = forward_step(b1, "no")
print(f"E1 P(lluvia1|par)={b1[0]:.3f} | E2 {b2[0]:.3f} | E3 {b2n[0]:.3f}")


## Reflexión

1. ¿Qué par de supuestos de independencia hacen que el filtrado cueste `O(|S|²)` por paso en lugar de crecer con la historia completa?
2. ¿Por qué la secuencia de Viterbi puede diferir de la secuencia de estados individualmente más probables paso a paso?
3. Si el modelo de emisión fuera perfecto (`P(e|x)` determinista y distinto por estado), ¿qué queda del problema? ¿Y si la transición fuera la identidad?
